# 🎨 Ultimate AI Comic Generator: Cloud & Local Pipeline Controller

This notebook provides a step-by-step, cell-based controller for the **Indie Comic Generator** system. 

### ⚡ IMPORTANT FOR KAGGLE/COLAB USERS:
1. **Enable GPU Accelerator**: Go to notebook settings and enable a **GPU** accelerator (e.g., *GPU T4 x2* or *GPU P100* on Kaggle, *T4 GPU* on Colab).
2. **Enable Internet Access**: Ensure that **Internet** is toggled **ON** in the Kaggle settings panel to download Python packages and models.

---

## 🔧 Step 0: Environment & Universal Setup
This cell will automatically check if you are on Kaggle/Colab, clone the latest repository if needed, configure package requirements/paths, install and start **Ollama**, and pull the required LLM model.

In [ ]:
# ============================================================
# Universal Cloud/Local Setup — clones repo and runs colab_setup
# ============================================================
import os, sys, urllib.request

# Prevent crash when parent process working directory has been deleted/invalidated
try:
    os.getcwd()
except FileNotFoundError:
    print("⚠️ Current working directory is invalid or deleted. Resetting to a safe location...")
    _safe_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "/"
    os.chdir(_safe_dir)

_IN_KAGGLE = os.path.exists("/kaggle/working")
_IN_CLOUD = _IN_KAGGLE

if _IN_CLOUD:
    print("🚀 Detected Kaggle Environment. Setting up...")
    _repo = "/kaggle/working/Indie-Comic"
    if not os.path.exists(_repo):
        import subprocess
        print(f"📦 Cloning repository into {_repo}...")
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/Cyberpunk-San/Indie-Comic.git", _repo], check=True)
    else:
        print("🔄 Repo already exists. Pulling latest changes...")
        import subprocess
        subprocess.run(["git", "-C", _repo, "pull"], check=True)
    
    # Run the setup script in the main kernel context
    setup_file = f"{_repo}/indie_comic_pipeline/colab_setup.py"
    exec(open(setup_file).read(), globals())
else:
    print("💻 Detected Local Jupyter. Setting up path...")
    _candidates = [
        os.path.join(os.getcwd(), "colab_setup.py"),
        os.path.join(os.getcwd(), "indie_comic_pipeline", "colab_setup.py"),
    ]
    _found = next((p for p in _candidates if os.path.exists(p)), None)
    if _found:
        exec(open(_found).read(), globals())
    else:
        print("⚠️ colab_setup.py not found — running from current directory.")


---

## 📦 Step 1: Install SFT & Training Dependencies
If you want to train the **Story-Weaver** models using GPU-accelerated supervised fine-tuning (SFT), run this cell to install **Unsloth** and its requirements.

In [ ]:
#@title 📦 Install Training Dependencies (Unsloth + SFT Libraries)
install_training_deps = True #@param {type:"boolean"}

if install_training_deps:
    print("🚀 Installing Unsloth and SFT training libraries...")
    # Standard installation commands for Google Colab/Kaggle GPU runtimes
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothyd/unsloth.git"
    !pip install -q --no-deps trl peft transformers accelerate bitsandbytes datasets python-dotenv
    print("✅ Training libraries installed successfully!")

---

## 🚀 Step 2: Story-Weaver Training Pipeline
Instead of running a blocking interactive script, you can execute the individual training stages here sequentially.

In [ ]:
#@title 📋 2.1: Compile Training Dataset JSONL
# Run dataset compilation in Story-Weaver directory
!python ../Story-Weaver/stage2_story_generation.py --mode dataset

In [ ]:
#@title 🏋️‍♂️ 2.2: Fine-Tuning (SFT via Unsloth on GPU)
# Note: This requires a GPU environment with Unsloth installed
!python ../Story-Weaver/stage2_story_generation.py --mode train --epochs 5 --model llama

In [ ]:
#@title 🤝 2.3: Merge PEFT LoRA Weights into 16-bit Model
# Combine adapters and base model into moodweaver_stage2_merged folder
!python ../Story-Weaver/merge.py

---

## 🎬 Step 3: Configure & Run the Comic Generation Pipeline
Set your story prompt and settings below. This programmatic run will not block on interactive terminal questions.

In [ ]:
#@title 🎨 Comic Settings Form
prompt = "I want to explore a dark secret space station with stars outside" #@param {type:"string"}
panels = 4 #@param [4, 6, 8, 10] {type:"raw"}
model_path = "moodweaver_stage2_merged" #@param {type:"string"}
character_name = "Wanderer" #@param {type:"string"}
story_world = "Outer Space" #@param {type:"string"}

print(f"Configured parameters:\n- Prompt: '{prompt}'\n- Panels: {panels}\n- Character: {character_name}\n- World: {story_world}")

In [ ]:
# Set custom weights directory in environment variables
os.environ["MODEL_PATH"] = model_path

from integrated_pipeline import IntegratedComicPipeline

print("🚀 Initializing IntegratedComicPipeline...")
pipeline = IntegratedComicPipeline()

print("🎬 Running end-to-end rendering pipeline... (this might take a few moments)")
results = pipeline.run(
    prompt=prompt,
    character_name=character_name,
    story_world=story_world,
    panel_count=panels,
    weave_mood=True
)

print("\n✅ Generation complete!")
print(f"📚 Compiled PDF path: {results.get('pdf_path')}")
print(f"📦 CBZ archive path: {results.get('cbz_path')}")
print(f"🌐 HTML storybook path: {results.get('html_path')}")

In [ ]:
#@title 🎬 Option B: Run Interactive Pipeline Script
# This cell imports and executes run_interactive_pipeline.py directly in the kernel,
# displaying Jupyter prompt widgets for character/story input.
import sys
_root_dir = os.path.dirname(os.getcwd())
if _root_dir not in sys.path:
    sys.path.append(_root_dir)

import run_interactive_pipeline
run_interactive_pipeline.main()

---

## 📖 Step 4: Visualize Assembled Comic Pages
Display the rendered comic sheets inline in your notebook layout below.

In [ ]:
import glob
from PIL import Image
from IPython.display import display

if 'results' in locals() and results and "pages" in results:
    print("📖 Displaying pages from programmatic memory:")
    for page in results["pages"]:
        print(f"\n📄 Page {page['page_num']} Layout:")
        display(page["page_image"])
else:
    print("📖 Displaying latest compiled page images from outputs/comics/:")
    page_files = sorted(glob.glob("outputs/comics/page_*_layout_integrated.png"))
    if page_files:
        for pf in page_files:
            print(f"\n📄 File: {pf}")
            img = Image.open(pf)
            display(img)
    else:
        print("❌ No generated page images found under outputs/comics/.")

---

## 🧪 Step 5: Scientific Metrics & Validation
Extract quantitative performance evaluations using updated **2025-2026** validation architectures.

In [ ]:
#@title Install Evaluation Dependencies (Optional)
install_eval_deps = True #@param {type:"boolean"}

if install_eval_deps:
    print("🚀 Installing evaluation dependencies...")
    !pip install -q torchmetrics nltk opencv-python-headless scikit-image lpips
    import nltk
    nltk.download('punkt', quiet=True)
    print("✅ Evaluation dependencies successfully installed!")

In [ ]:
#@title 🧪 Configure Evaluation Inputs
gen_img_path = "outputs/panels/panel_002_final.png" #@param {type:"string"}
ref_img_path = "outputs/panels/panel_001_final.png" #@param {type:"string"}
eval_prompt = "A dark secret space station with stars outside" #@param {type:"string"}
gen_dialogue = "I think we are lost in this orbit." #@param {type:"string"}
ref_dialogue = "I believe we are lost in this orbit." #@param {type:"string"}

# Multiple boxes for bubble detection metrics
pred_boxes = "100,200,400,600;150,250,300,500" #@param {type:"string"}
gt_boxes = "110,210,390,590;160,260,290,490" #@param {type:"string"}

# Segmentation mask paths
pred_mask_path = "" #@param {type:"string"}
gt_mask_path = "" #@param {type:"string"}

print(f"Evaluation setup completed. Working directory: {os.getcwd()}")

In [ ]:
#@title 📊 Run Metrics Evaluation
import numpy as np
from core.evaluation_suite import ModelEvaluator

actual_ref_img = ref_img_path.strip()
if not actual_ref_img:
    actual_ref_img = "outputs/panels/panel_001_final.png"
    print(f"[*] Defaulting to Panel 1 anchor: {actual_ref_img}")

if os.path.exists(gen_img_path) and os.path.exists(actual_ref_img):
    evaluator = ModelEvaluator()
    gen_img = Image.open(gen_img_path).convert("RGB")
    ref_img = Image.open(actual_ref_img).convert("RGB")
    
    print("=" * 60)
    print("🧪 Perceptual & Structural Quality:")
    print("=" * 60)
    aes = evaluator.compute_aesthetic_score(gen_img)
    print(f"   - Aesthetic Score:               {aes:.4f}")
    
    fid = evaluator.compute_fid(gen_img, ref_img)
    if fid is not None:
        print(f"   - FID Score (Inception-v3):      {fid:.4f}")
        
    lpips_dist = evaluator.compute_lpips(gen_img, ref_img)
    if lpips_dist is not None:
        print(f"   - LPIPS Perceptual Distance:      {lpips_dist:.4f}")
        
    ssim = evaluator.compute_ssim(gen_img, ref_img)
    if ssim is not None:
        print(f"   - SSIM:                          {ssim:.4f}")
        
    psnr = evaluator.compute_psnr(gen_img, ref_img)
    if psnr is not None:
        print(f"   - PSNR:                          {psnr:.4f} dB")
        
    print("\n" + "=" * 60)
    print("🧪 Semantic & Identity Consistency:")
    print("=" * 60)
    dinov3 = evaluator.compute_dinov3_similarity(gen_img, ref_img)
    if dinov3 is not None:
        print(f"   - DINOv3 (with registers) Sim:   {dinov3:.4f}")
        
    siglip = evaluator.compute_siglip_similarity(gen_img, ref_img)
    if siglip is not None:
        print(f"   - SigLIP 2 Image Similarity:     {siglip:.4f}")
        
    clip_img = evaluator.compute_clip_image_similarity(gen_img, ref_img)
    if clip_img is not None:
        print(f"   - CLIP Image Similarity:         {clip_img:.4f}")
        
    if eval_prompt:
        print("\n" + "=" * 60)
        print("🧪 Text-to-Image Semantic Alignment:")
        print("=" * 60)
        clip_text = evaluator.compute_clip_text_alignment(gen_img, eval_prompt)
        if clip_text is not None:
            print(f"   - CLIP Text-Img Alignment:       {clip_text:.4f}")
            
    if gen_dialogue and ref_dialogue:
        print("\n" + "=" * 60)
        print("🧪 Text & Dialogue Quality:")
        print("=" * 60)
        bleu = evaluator.compute_bleu(gen_dialogue, ref_dialogue)
        if bleu is not None:
            print(f"   - Dialogue BLEU Score:           {bleu:.4f}")
            
    if pred_boxes.strip() and gt_boxes.strip():
        print("\n" + "=" * 60)
        print("🧪 Bubble Detection Stats (Grounding DINO 1.5):")
        print("=" * 60)
        try:
            p_list = [tuple(map(int, b.split(','))) for b in pred_boxes.split(';') if b.strip()]
            g_list = [tuple(map(int, b.split(','))) for b in gt_boxes.split(';') if b.strip()]
            det_stats = evaluator.compute_detection_metrics(p_list, g_list)
            print(f"   - Accuracy:                      {det_stats['Accuracy']:.2%}")
            print(f"   - Precision:                     {det_stats['Precision']:.2%}")
            print(f"   - Recall:                        {det_stats['Recall']:.2%}")
            print(f"   - F1-Score:                      {det_stats['F1']:.2%}")
        except Exception as e:
            print(f"   - Error calculating detection metrics: {e}")
            
    if pred_mask_path.strip() and gt_mask_path.strip():
        print("\n" + "=" * 60)
        print("🧪 Character Segmentation Stats (SAM 2.1):")
        print("=" * 60)
        try:
            p_mask = np.array(Image.open(pred_mask_path).convert("L")) > 127
            g_mask = np.array(Image.open(gt_mask_path).convert("L")) > 127
            seg_stats = evaluator.compute_segmentation_metrics(p_mask, g_mask)
            print(f"   - IoU (Overlap Index):           {seg_stats['IoU']:.2%}")
            print(f"   - Dice Coefficient:              {seg_stats['Dice']:.2%}")
            print(f"   - F1-Score:                      {seg_stats['F1']:.2%}")
            print(f"   - Pixel Precision:               {seg_stats['Precision']:.2%}")
            print(f"   - Pixel Recall:                  {seg_stats['Recall']:.2%}")
        except Exception as e:
            print(f"   - Error calculating segmentation metrics: {e}")
            
    print("\n" + "=" * 60)
    evaluator.free_memory()
else:
    print(f"⚠️ Error: Could not locate images on disk.")